# Sephora Dataset — Exploratory Data Analysis

Run this notebook **before** training to understand the data.

Open it by running:
```
jupyter notebook notebooks/exploration.ipynb
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

products = pd.read_csv('../input/product_info.csv')
reviews  = pd.read_csv('../input/reviews_0-250.csv')

print(f'Products: {len(products):,}')
print(f'Reviews:  {len(reviews):,}')
products.head()

In [ ]:
# Basic info
print('=== Products shape:', products.shape)
print('\n=== Columns:', products.columns.tolist())
print('\n=== Missing values:')
print(products.isnull().sum()[products.isnull().sum() > 0])
print('\n=== Data types:')
print(products.dtypes)

In [ ]:
# Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

products['rating'].hist(bins=30, ax=axes[0], color='#c97b7b', edgecolor='white')
axes[0].set_title('Product Rating Distribution')
axes[0].set_xlabel('Rating')

products['loves_count'].clip(upper=10000).hist(bins=40, ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Loves Count Distribution (clipped at 10k)')
axes[1].set_xlabel('Number of people who saved it')

plt.tight_layout()
plt.show()

In [ ]:
# Target label: worth_it
products['worth_it'] = ((products['rating'] >= 4.0) & (products['loves_count'] >= 500)).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

products['worth_it'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[0].set_title('Class Balance')
axes[0].set_xticklabels(['Overhyped (0)', 'Worth It (1)'], rotation=0)

products.groupby('primary_category')['worth_it'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='#c97b7b')
axes[1].set_title('Worth-It Rate by Category')
axes[1].set_xlabel('Fraction of products rated worth-it')

plt.tight_layout()
plt.show()

In [ ]:
# Price vs rating
plt.figure(figsize=(10, 5))
sample = products.dropna(subset=['price_usd','rating']).sample(min(2000, len(products)), random_state=42)
plt.scatter(sample['price_usd'], sample['rating'],
            c=sample['worth_it'], cmap='RdYlGn', alpha=0.5, edgecolors='none', s=20)
plt.colorbar(label='Worth It (1=yes)')
plt.xlabel('Price (USD)')
plt.ylabel('Rating')
plt.title('Price vs Rating — coloured by Worth-It label')
plt.axhline(4.0, color='gray', linestyle='--', lw=0.8)
plt.xlim(0, 300)
plt.show()

In [ ]:
# Correlation heatmap
num_cols = ['price_usd', 'rating', 'loves_count', 'reviews',
            'limited_edition', 'new', 'online_only', 'sephora_exclusive', 'out_of_stock']
corr = products[[c for c in num_cols if c in products.columns]].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Top brands by average rating
brand_stats = (products.groupby('brand_name')
               .agg(avg_rating=('rating','mean'), n_products=('rating','count'))
               .query('n_products >= 5')
               .sort_values('avg_rating', ascending=False)
               .head(20))

brand_stats['avg_rating'].plot(kind='barh', figsize=(10,7), color='#c97b7b')
plt.title('Top 20 Brands by Average Rating (min 5 products)')
plt.xlabel('Average Rating')
plt.tight_layout()
plt.show()